In [24]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint,HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
from dotenv import load_dotenv

In [ ]:
load_dotenv()

llm=HuggingFaceEndpoint(repo_id='openai/gpt-oss-120b', task='text-generation')

model=ChatHuggingFace(llm=llm)

embedding_model=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

parser=StrOutputParser()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6218.35it/s]


In [ ]:
##GENERATE YOUTUBE TRANSCIPT
url=input()
if '&' in url:
  url=url.split('&')[0]
video_id=url.split('=')[1]

ytt_api=YouTubeTranscriptApi()
video_id='lIo9FcrljDk'
try:
  fetched_transcript=ytt_api.fetch(video_id=video_id)
  transcript=' '.join(snippet.text for snippet in fetched_transcript)
except TranscriptsDisabled:
  print('No captions available for thi video')

transcript

"ANDREW HUBERMAN: Welcome to\nHuberman Lab Essentials, where we revisit past episodes for\nthe most potent and actionable science based tools for mental\nhealth, physical health, and performance. [MUSIC PLAYING] I'm Andrew Huberman,\nand I'm a professor of neurobiology\nand ophthalmology at Stanford School of Medicine. Today's podcast episode\nis all about sleep. We're also going to talk about\nthe mirror image of sleep, which is wakefulness. Now, these two phases of our\nlife, sleep and wakefulness govern everything about our\nmental and physical health. And we're not just going to talk\nabout what's useful about sleep. We're also going to talk about\nhow to get better at sleeping, and that will include\nhow to get better at falling asleep,\ntiming your sleep and accessing better\nsleep quality. In doing so, we're\nalso going to discuss how to get more focused\nand alert in wakefulness. So if you're like most\npeople, which includes me, you have some\nchallenges with sleep. At least e

In [27]:
##SPLITTING THE TRANSCRIPT INTO CHUNKS
splitter= RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=10)
chunks=splitter.create_documents([transcript])
chunks

[Document(metadata={}, page_content="ANDREW HUBERMAN: Welcome to\nHuberman Lab Essentials, where we revisit past episodes for\nthe most potent and actionable science based tools for mental\nhealth, physical health, and performance. [MUSIC PLAYING] I'm Andrew Huberman,\nand I'm a professor of neurobiology\nand ophthalmology at Stanford School of Medicine. Today's podcast episode\nis all about sleep. We're also going to talk about\nthe mirror image of sleep, which is wakefulness. Now, these two phases of our"),
 Document(metadata={}, page_content="life, sleep and wakefulness govern everything about our\nmental and physical health. And we're not just going to talk\nabout what's useful about sleep. We're also going to talk about\nhow to get better at sleeping, and that will include\nhow to get better at falling asleep,\ntiming your sleep and accessing better\nsleep quality. In doing so, we're\nalso going to discuss how to get more focused\nand alert in wakefulness. So if you're like most\n

In [28]:
##STORE THE CHUNKS INTO VECTOR STORE
vector_store=Chroma.from_documents(documents=chunks, embedding=embedding_model, collection_name='sample', persist_directory='chroma_db')
vector_store.get()

{'ids': ['f7a2bc1a-3024-4e45-baa3-8f1f46860806',
  '60e2c1a4-43bc-4bb0-aa9f-e84f8c3636ee',
  '15af457d-9f0f-497b-b9dc-0922350e9a2c',
  'd1f5de9b-f57e-4d6f-996d-1376571fc43f',
  '9991c8c8-30ac-4a2d-9c9c-8497d7dfbb44',
  '2167476f-6e5f-4452-b72d-fee68450c4aa',
  '9563bb60-0330-4d52-bb7a-ad2f269ca23e',
  '2b199f50-3043-4278-adb2-13b86e074630',
  '4d7a3ecb-b267-4f35-8074-a6ffd1d37e93',
  '26ce0630-793d-46e1-a817-772aee4965f1',
  '5460420c-aab6-4126-bb94-2ff9cc264a9e',
  '09620955-d4e9-419d-b8a0-806f6d750126',
  'd66d6ffd-1853-4ddd-bd78-8921e9272e45',
  '3ac1e876-56ae-46e9-9e9e-e3d896841ebc',
  '4d5cdcf4-17e9-4772-bef1-f78430680b6d',
  '40860d24-cc9c-4d4c-95ef-1aa39bfcb4ff',
  '6a0e353a-3cd5-4c5b-b62c-dcb7ccfcd7ff',
  '01869e0e-c32c-47b3-b9b2-baadfa4481eb',
  '30fa40a5-45c6-4133-bc78-6d975c97cb68',
  '1cb78412-fea6-4990-8f0e-6c43da1e5773',
  'e1fc7d5c-c1b4-4fe8-8c00-6da7b09e407d',
  'aab1d5fe-e727-4c9d-b4b3-bb76d61c5683',
  'e9472cfb-f157-4c50-93b8-6530f8155be2',
  'e78427a5-aa0a-45ec-81d2-

In [29]:
##BUILDING A RETRIEVER
query='what is dopamine'
retriever=vector_store.as_retriever(search_type='similarity',search_kwargs={'k': 2})
retrieved_docs=retriever.invoke(query)

In [30]:
##AUGMENTATION--> COMBINING QUERY AND CONTEXT TO FORM THE PROMPT
prompt=PromptTemplate(template='''You are a helpful assistant
answer only from the context provided.
if you dont know the answer just say no
{context}
question: {question}''', input_variables=['context', 'question'])

def get_context(retrieved_docs):
  context='\n\n'.join(doc.page_content for doc in retrieved_docs)
  return context

context=get_context(retrieved_docs)

final_prompt=prompt.invoke({
  'context': context,
  'question': query
})

final_prompt

StringPromptValue(text="You are a helpful assistant\nanswer only from the context provided.\nif you dont know the answer just say no\nbetween 11:00 PM and 4:00 AM approximately, suppresses\nthe release of dopamine. This neuromodulator\nthat makes us feel good is sort of an endogenous\nantidepressant and can inhibit learning\nand create all sorts of other detrimental effects. It does this through a\nmechanism for those of you who want to the neural pathways\nthat involves light to the eyes. That's then signaled to a\nstructure called the habenula. When that habenula\ngets activated, it's actually called the\n\nbetween 11:00 PM and 4:00 AM approximately, suppresses\nthe release of dopamine. This neuromodulator\nthat makes us feel good is sort of an endogenous\nantidepressant and can inhibit learning\nand create all sorts of other detrimental effects. It does this through a\nmechanism for those of you who want to the neural pathways\nthat involves light to the eyes. That's then signaled t

In [35]:
##SENDING PROMPT TO LLM

answer=model.invoke(final_prompt)
print(answer)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ace46ed-60e7fa743910153f107a092d;a6735fa7-7db2-498a-a597-31252d33d304)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.

In [36]:
from langchain_core.runnables.base import RunnableSequence, RunnableParallel, RunnableLambda
from langchain_core.runnables.passthrough import RunnablePassthrough

In [37]:
##CREATING A CHAIN
retriever_chain= RunnableParallel({
  'context': retriever | RunnableLambda(get_context), 
  'question': RunnablePassthrough()
})

main_chain= retriever_chain | prompt | model | parser
main_chain.invoke(query)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6ace4b26-247702b12d1479032cc192fb;64388021-b71a-46a0-a4c4-010a97772651)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have no remaining credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get monthly included credits.